# 08 — Extraction Reliability: Blind Second Coding

MallWorld audit (2026-10)

---

## Purpose

The archived thesis stated that "extraction reliability was validated through manual review". No record of such a review exists in the repository. This notebook measures reproducibility directly, as was done for the NDE project.

**Design.** A blind second coder (Claude) recoded 119 dreams from their text: `validation/second_coder_codes.jsonl`, with procedure and conventions in `validation/README.md`. The codes were committed (`498e86f4`) before this notebook was written.
- **Set A** is a random sample of 50 primary dreams (253 locations). It covers atmosphere, vertical level, light, affect, threat entities and deceased persons.
- **Set B** contains the 69 dreams behind the pre-registered tests P1 (animals) and P3 (deceased). It covers atmosphere, animals and deceased persons per location.

**What would count against the extraction.**
- κ below 0.40 on a field that a finding rests on.
- A pre-registered verdict that reverses when the second coder's codes are used.

Agreement is reported as Cohen's κ, with a 95% CI from a bootstrap over dreams (2,000 resamples). It is computed on all locations, including `not_mentioned`, and separately on locations both coders coded. The second coder is not ground truth (README, Limitations).

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, fisher_exact
from statsmodels.stats.multitest import multipletests

MW_ROOT = Path.cwd().parent
sys.path.insert(0, str(MW_ROOT / "scripts"))
import mallworld_dataset as mw  # verified loader: schema-checked fields, populations, cluster keys

OUTPUT_DIR = MW_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

posts = mw.load_posts()
primary = posts[posts["primary"]].copy()
print(f"Posts: {len(posts):,} | primary dream reports: {len(primary):,} from {primary['author'].nunique():,} authors")

import re
from sklearn.metrics import cohen_kappa_score
VAL = MW_ROOT / "validation"
codes = [json.loads(l) for l in open(VAL / "second_coder_codes.jsonl")]
L = mw.load_locations(); E = mw.load_entities()

# Re-draw the samples and check they match the coded dreams
idsA = np.random.default_rng(20261008).choice(np.sort(primary["post_id"].values), 50, replace=False)
assert set(idsA) == {c["post_id"] for c in codes if c["set"] == "A"}

GENTLE = set("cat cats kitten kittens kitty dog dogs puppy puppies sheep lamb lambs horse horses pony cow cows cattle calf deer fawn "
             "rabbit rabbits bunny bunnies bird birds dove doves chicken chickens duck ducks swan swans fish goldfish dolphin dolphins butterfly "
             "butterflies squirrel squirrels hamster turtle turtles owl owls".split())
NOXIOUS = set("snake snakes serpent serpents spider spiders rat rats mouse mice cockroach cockroaches roach roaches insect insects bug bugs "
              "ant ants wasp wasps hornet hornets scorpion scorpions worm worms maggot maggots leech leeches wolf wolves shark sharks crocodile "
              "crocodiles alligator alligators bear bears lion lions tiger tigers hound hounds hyena hyenas vulture vultures predator predators".split())
re_g = re.compile(r"\b(?:" + "|".join(sorted(GENTLE)) + r")\b"); re_n = re.compile(r"\b(?:" + "|".join(sorted(NOXIOUS)) + r")\b")
clean = lambda s: re.sub(r"\bhot ?dogs?\b", " ", (s or "").lower())
E["text"] = (E["entity_description"].fillna("") + " " + E["name_or_relation"].fillna("")).map(clean)
et = E.groupby(["post_id", "location_id"])["text"].apply(" ".join)
L["t"] = [et.get((p, l), "") + " " + clean(d) for p, l, d in zip(L["post_id"], L["location_id"], L["raw_description"])]
L["g1"] = L["t"].str.contains(re_g); L["n1"] = L["t"].str.contains(re_n)
L["cls1"] = np.select([L["n1"] & ~L["g1"], L["g1"] & ~L["n1"], L["g1"] & L["n1"]], ["noxious", "gentle", "both"], "none")
types_at = E.groupby(["post_id", "location_id"])["entity_type"].apply(set)
L["dec1"] = [int("deceased" in types_at.get((p, l), set())) for p, l in zip(L["post_id"], L["location_id"])]
L["living_known1"] = [int("deceased" not in s and bool(s & {"known_person", "family_member", "friend"}))
                      for s in (types_at.get((p, l), set()) for p, l in zip(L["post_id"], L["location_id"]))]

def cls_of(animals):
    toks = set(w for a in animals for w in re.findall(r"[a-z]+", a.lower()))
    g, n = bool(toks & GENTLE), bool(toks & NOXIOUS)
    return "both" if g and n else "noxious" if n else "gentle" if g else "none"

rows = []
for c in codes:
    for lid, v in c["locations"].items():
        rows.append({"set": c["set"], "post_id": c["post_id"], "location_id": lid, "image_only": c.get("image_only", False),
                     "atm2": v["atm"], "vert2": v.get("vert"), "light2": v.get("light"), "affect2": v.get("affect"),
                     "cls2": cls_of(v.get("animals", [])) if c["set"] == "B" else None, "dec2": v.get("deceased_here")})
S = pd.DataFrame(rows)
for col, field in [("atm2", "atmosphere"), ("vert2", "vertical"), ("light2", "light"), ("affect2", "affective_response")]:
    mw.check_values(field, S[col].dropna().unique())
M = S.merge(L, on=["post_id", "location_id"], how="left", indicator=True)
assert (M["_merge"] == "both").all(), M.loc[M["_merge"] != "both", ["post_id", "location_id"]]
n_ext = L[L["post_id"].isin(S["post_id"])].groupby("post_id").size()
assert (S.groupby(["set", "post_id"]).size().droplevel(0).groupby(level=0).max().reindex(n_ext.index) == n_ext).all()
M["val2"] = M["atm2"].map(mw.ATMOSPHERE_VALENCE)
print(f"Coded locations: set A {int((M['set'] == 'A').sum())} in {M.loc[M['set'] == 'A', 'post_id'].nunique()} dreams "
      f"({int(M.loc[M['set'] == 'A', 'image_only'].sum())} locations in image-only posts); set B {int((M['set'] == 'B').sum())} in {M.loc[M['set'] == 'B', 'post_id'].nunique()} dreams")

rng = np.random.default_rng(8)
def kappa_ci(df, a, b, B=2000):
    d = df.dropna(subset=[a, b])
    k = cohen_kappa_score(d[a].astype(str), d[b].astype(str))
    groups = {p: g for p, g in d.groupby("post_id")}; ids = np.array(list(groups))
    ks = []
    for _ in range(B):
        bd = pd.concat([groups[i] for i in rng.choice(ids, len(ids))])
        if bd[a].nunique() > 1 or bd[b].nunique() > 1:
            ks.append(cohen_kappa_score(bd[a].astype(str), bd[b].astype(str)))
    lo, hi = np.nanpercentile(ks, [2.5, 97.5])
    return k, lo, hi, len(d), d["post_id"].nunique(), (d[a].astype(str) == d[b].astype(str)).mean()
def show(label, df, a, b):
    k, lo, hi, n, C, agree = kappa_ci(df, a, b)
    print(f"{label:55s} kappa = {k:.2f} [{lo:.2f}-{hi:.2f}]  agreement {100 * agree:.1f}%  (n = {n}, dreams = {C})")
    return k

Posts: 3,732 | primary dream reports: 1,918 from 1,303 authors


Coded locations: set A 253 in 50 dreams (8 locations in image-only posts); set B 474 in 69 dreams


## R1 — Location codes (set A, text-only posts)

In [2]:
A = M[(M["set"] == "A") & ~M["image_only"]].copy()
A["val1c"] = A["valence"].map({-1: "neg", 0: "neu", 1: "pos"}).fillna("nm"); A["val2c"] = A["val2"].map({-1: "neg", 0: "neu", 1: "pos"}).fillna("nm")
A.loc[A["atmosphere"] == "nostalgic", "val1c"] = "nostalgic"; A.loc[A["atm2"] == "nostalgic", "val2c"] = "nostalgic"
results = {}
for lab, a, b in [("atmosphere (10 values incl. not_mentioned)", "atmosphere", "atm2"), ("atmosphere valence (neg/neu/pos/nostalgic/not coded)", "val1c", "val2c")]:
    results[lab] = show(lab, A, a, b)
both = A[(A["val1c"].isin(["neg", "neu", "pos"])) & (A["val2c"].isin(["neg", "neu", "pos"]))].assign(n1=lambda d: d["val1c"] == "neg", n2=lambda d: d["val2c"] == "neg")
results["negative vs not, both coded"] = show("negative vs not, locations both coders rated", both, "n1", "n2")
results["valence, both coded"] = show("valence (neg/neu/pos), locations both coders rated", both, "val1c", "val2c")
A["atm_coded1"] = A["atmosphere"] != "not_mentioned"; A["atm_coded2"] = A["atm2"] != "not_mentioned"
print(f"Atmosphere coded at all: GPT-5.2 {100 * A['atm_coded1'].mean():.1f}% vs second coder {100 * A['atm_coded2'].mean():.1f}% of {len(A)} locations")
results["atmosphere coded vs not"] = show("atmosphere coded at all (yes/no)", A, "atm_coded1", "atm_coded2")
r1 = A[A["val1c"].isin(["neg", "neu", "pos"])]
sh_both = (r1.loc[r1["val2c"].isin(["neg", "neu", "pos"]), "val1c"] == "neg").mean(); sh_extra = (r1.loc[r1["val2c"] == "nm", "val1c"] == "neg").mean()
print(f"GPT-5.2 negative share where the second coder also rated: {100 * sh_both:.1f}%; where only GPT-5.2 rated: {100 * sh_extra:.1f}% "
      f"({int((r1['val2c'] == 'nm').sum())} locations)")
for lab, a, b in [("vertical (7 values)", "vertical", "vert2"), ("light (9 values)", "light", "light2"), ("affective response (10 values)", "affective_response", "affect2")]:
    results[lab] = show(lab, A, a, b)
    print(f"{'':55s} coded: GPT-5.2 {100 * (A[a] != 'not_mentioned').mean():.1f}% vs second coder {100 * (A[b] != 'not_mentioned').mean():.1f}%")
    bc = A[(A[a] != "not_mentioned") & (A[b] != "not_mentioned")]
    if len(bc) >= 10:
        show(f"   {lab.split(' (')[0]}, locations both coders coded", bc, a, b)
A["lvl1"] = np.sign(A["vertical_level"]).map({-1: "below", 0: "ground", 1: "above"}).fillna("nm")
A["lvl2"] = A["vert2"].map({"lowest": "below", "lower": "below", "ground": "ground", "upper": "above", "uppermost": "above"}).fillna("nm")
results["vertical below/ground/above"] = show("vertical as below/ground/above/not coded", A, "lvl1", "lvl2")
print("\nAtmosphere cross-table (rows GPT-5.2, columns second coder):")
print(pd.crosstab(A["val1c"], A["val2c"]).to_string())

atmosphere (10 values incl. not_mentioned)              kappa = 0.36 [0.27-0.47]  agreement 56.7%  (n = 245, dreams = 47)


atmosphere valence (neg/neu/pos/nostalgic/not coded)    kappa = 0.42 [0.30-0.53]  agreement 64.9%  (n = 245, dreams = 47)


negative vs not, locations both coders rated            kappa = 0.74 [0.46-0.94]  agreement 93.7%  (n = 63, dreams = 33)


valence (neg/neu/pos), locations both coders rated      kappa = 0.75 [0.48-0.94]  agreement 93.7%  (n = 63, dreams = 33)
Atmosphere coded at all: GPT-5.2 58.0% vs second coder 29.0% of 245 locations


atmosphere coded at all (yes/no)                        kappa = 0.38 [0.25-0.52]  agreement 66.9%  (n = 245, dreams = 47)
GPT-5.2 negative share where the second coder also rated: 82.5%; where only GPT-5.2 rated: 58.9% (73 locations)


...

light (9 values)                                        kappa = 0.61 [0.46-0.74]  agreement 89.0%  (n = 245, dreams = 47)
                                                        coded: GPT-5.2 20.0% vs second coder 10.6%


   light, locations both coders coded                   kappa = 0.90 [0.76-1.00]  agreement 92.0%  (n = 25, dreams = 19)


affective response (10 values)                          kappa = 0.35 [0.25-0.46]  agreement 62.9%  (n = 245, dreams = 47)
                                                        coded: GPT-5.2 49.4% vs second coder 19.2%


   affective response, locations both coders coded      kappa = 0.67 [0.53-0.81]  agreement 75.0%  (n = 44, dreams = 28)


vertical as below/ground/above/not coded                kappa = 0.51 [0.39-0.64]  agreement 74.3%  (n = 245, dreams = 47)

Atmosphere cross-table (rows GPT-5.2, columns second coder):
val2c      neg  nm  nostalgic  pos
val1c                             
neg         52  43          0    0
neu          3  21          0    0
nm           3  98          0    2
nostalgic    0   3          2    1
pos          1   9          0    7


### Findings — R1

**The two coders agree on the value, and disagree on whether the text carries one.**

| Field | Coded by GPT-5.2 | Coded by second coder | κ, all locations | κ, both coded (n) |
|---|---|---|---|---|
| Atmosphere (valence) | 58.0% | 29.0% | 0.42 | **0.75** (63) |
| Atmosphere, negative vs not | | | | **0.74** (63) |
| Vertical level | 51.0% | 20.8% | 0.42 | **0.84** (49) |
| Light | 20.0% | 10.6% | 0.61 | **0.90** (25) |
| Affective response | 49.4% | 19.2% | 0.35 | **0.67** (44) |

Data: 245 text-only locations in 47 dreams, set A. CIs are in the cell output.

**What GPT-5.2 adds.**
- GPT-5.2 rates atmosphere, level and affect at about twice as many locations.
- It infers them from events ("chased" → threatening) and from location types (a subway → lower), which the extraction prompt invites. The second coder required an explicit statement.
- GPT-5.2's additional atmosphere ratings are *less* often negative (58.9% of 73) than its ratings where the second coder also found explicit affect (82.5%).

**Consequences.**
- **Values are reproducible; prevalence is convention-dependent.** Wherever both coders judged that the text describes the atmosphere, level or light, the value is reproducible (κ 0.74–0.90, comparable to the better NDE fields). Prevalence figures are convention-dependent. Examples are "64% of locations are negative", "34% have a vertical level" and "half the locations carry an affect". They describe what GPT-5.2 inferred, not what dreamers stated.
- **Associations rest on the inferred ratings too.** The estimates in notebooks 02–07 use them as well. Agreement where both coders rated is high, but the inferred ratings were not checked against an independent judgement of the same inference.
- **Affect is the weakest field** (κ 0.67 where both coded). It is the basis of the corrected R4 result in notebook 04, which is labelled exploratory.

## R2 — Dream-level entity codes (set A)

In [3]:
Ad = pd.DataFrame([{"post_id": c["post_id"], "threat2": c["threat"], "dec2": c["deceased"], "image_only": c["image_only"]} for c in codes if c["set"] == "A"])
ent_types = E.groupby("post_id")["entity_type"].apply(set)
Ad["threat1"] = Ad["post_id"].map(lambda p: int("threat" in ent_types.get(p, set())))
Ad["dec1"] = Ad["post_id"].map(lambda p: int("deceased" in ent_types.get(p, set())))
hostile_any = E[E["demeanor"].isin(["hostile", "threatening"])].groupby("post_id").size()
Ad["hostile1"] = Ad["post_id"].map(lambda p: int(("threat" in ent_types.get(p, set())) or hostile_any.get(p, 0) > 0))
Ad = Ad[~Ad["image_only"]]
Ad["hostile2"] = Ad["threat2"]
for a, b in [("threat1", "threat2"), ("hostile1", "hostile2"), ("dec1", "dec2")]:
    tab = pd.crosstab(Ad[a], Ad[b])
    print(f"{a[:-1]}: kappa = {cohen_kappa_score(Ad[a], Ad[b]):.2f}; GPT-5.2 {int(Ad[a].sum())} dreams, second coder {int(Ad[b].sum())} of {len(Ad)}")
    print(tab.to_string())

threat: kappa = 0.50; GPT-5.2 7 dreams, second coder 13 of 47
threat2   0  1
threat1       
0        33  7
1         1  6
hostile: kappa = 0.71; GPT-5.2 17 dreams, second coder 13 of 47
hostile2   0   1
hostile1        
0         29   1
1          5  12
dec: kappa = nan; GPT-5.2 0 dreams, second coder 0 of 47
dec2   0
dec1    
0     47


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


### Findings — R2

- **Threat.** The `threat` entity type matches the second coder's "hostile entity present" only moderately: κ = 0.50 (GPT-5.2 7 dreams, second coder 13 of 47). GPT-5.2 often types hostile cult members, guards or crowds as `authority`, `crowd` or `other` with a hostile demeanor. Counting any entity with hostile or threatening demeanor raises agreement to κ = 0.71. Findings that use the `threat` type alone (notebook 03) therefore cover only part of the hostile encounters.
- **Deceased persons.** No dream in set A contains a deceased person, under either coder. Set B tests this field.

## R3 — The pre-registered tests under the second coder (set B)

P1 and P3 rest on 33 noxious-animal and 22 deceased-person locations. Rule 12 (CLAUDE.md §4) requires a coder-sensitivity check for results built on few events.

In [4]:
B_ = M[M["set"] == "B"].copy()
B_["dec2"] = B_["dec2"].astype(int)
assert B_["post_id"].nunique() == 69
print(f"Set B: {len(B_)} locations in {B_['post_id'].nunique()} dreams")
B_["val1c"] = B_["valence"].map({-1: "neg", 0: "neu", 1: "pos"}).fillna("nm"); B_["val2c"] = B_["val2"].map({-1: "neg", 0: "neu", 1: "pos"}).fillna("nm")
show("atmosphere valence incl. not coded", B_, "val1c", "val2c")
bb = B_[B_["val1c"].isin(["neg", "neu", "pos"]) & B_["val2c"].isin(["neg", "neu", "pos"])].assign(n1=lambda d: d["val1c"] == "neg", n2=lambda d: d["val2c"] == "neg")
show("negative vs not, both rated", bb, "n1", "n2")
show("deceased person at the location", B_, "dec1", "dec2")
show("animal class at the location (none/gentle/noxious/both)", B_, "cls1", "cls2")
print(pd.crosstab(B_["cls1"], B_["cls2"]).to_string())

B_["neg1"] = (B_["valence"] == -1).astype(float).where(B_["valence"].notna()); B_["neg2"] = (B_["val2"] == -1).astype(float).where(B_["val2"].notna())
def rate(d, col):
    d = d.dropna(subset=[col]); return f"{100 * d[col].mean():.1f}% ({len(d)} locations, {d['post_id'].nunique()} dreams)"
def p1(df, cls_col, neg_col, label):
    d = df[df[cls_col].isin(["noxious", "gentle"])].dropna(subset=[neg_col]).assign(nox=lambda x: (x[cls_col] == "noxious").astype(int), y=lambda x: x[neg_col])
    r = mw.clustered_logit(d, "y ~ nox + log_words"); t = mw.odds_ratio_table(r, ["nox"]).iloc[0]
    p_one = r.pvalues["nox"] / 2 if r.params["nox"] > 0 else 1 - r.pvalues["nox"] / 2
    print(f"  {label:48s} noxious {rate(d[d.nox == 1], 'y')}, gentle {rate(d[d.nox == 0], 'y')}; OR {t.OR:.2f} [{t.lo:.2f}-{t.hi:.2f}], one-sided p = {p_one:.3g}")
print("\nP1 within set B (all 27 noxious dreams, 27 of 81 gentle dreams):")
p1(B_, "cls1", "neg1", "GPT-5.2 animals, GPT-5.2 atmosphere")
p1(B_, "cls2", "neg1", "second-coder animals, GPT-5.2 atmosphere")
p1(B_, "cls1", "neg2", "GPT-5.2 animals, second-coder atmosphere")
p1(B_, "cls2", "neg2", "second-coder animals, second-coder atmosphere")

print("\nP3: deceased locations")
print(f"  GPT-5.2 deceased locations: {int(B_['dec1'].sum())} in {B_.loc[B_['dec1'] == 1, 'post_id'].nunique()} dreams; second coder: {int(B_['dec2'].sum())} in {B_.loc[B_['dec2'] == 1, 'post_id'].nunique()} dreams; both: {int((B_['dec1'] & B_['dec2']).sum())}")
Lp = L.assign(neg=(L["valence"] == -1).astype(float).where(L["valence"].notna()))
lk = Lp[Lp["living_known1"] == 1].dropna(subset=["neg"])
def p3(dec_col, neg_col, label):
    dd = B_[B_[dec_col] == 1].dropna(subset=[neg_col]).assign(neg=lambda d: d[neg_col], dec=1)
    d = pd.concat([dd[["post_id", "neg", "dec", "log_words"]], lk[~lk["post_id"].isin(dd["post_id"])].assign(dec=0)[["post_id", "neg", "dec", "log_words"]]])
    r = mw.clustered_logit(d, "neg ~ dec + log_words"); t = mw.odds_ratio_table(r, ["dec"]).iloc[0]
    p_one = r.pvalues["dec"] / 2 if r.params["dec"] < 0 else 1 - r.pvalues["dec"] / 2
    print(f"  {label:48s} deceased {rate(dd, 'neg')} vs living known (GPT-5.2) {100 * lk['neg'].mean():.1f}%; OR {t.OR:.2f} [{t.lo:.2f}-{t.hi:.2f}], one-sided p = {p_one:.3g}")
p3("dec1", "neg1", "GPT-5.2 deceased, GPT-5.2 atmosphere")
p3("dec2", "neg1", "second-coder deceased, GPT-5.2 atmosphere")
p3("dec1", "neg2", "GPT-5.2 deceased, second-coder atmosphere")
print("  (The second coder rates only locations with explicit affect, which GPT-5.2 also rates as negative more often than the locations only it rates (R1);")
print("   mixing coders in this row therefore works against P3, not for it.)")
for nm, mask in [("noxious (GPT-5.2)", B_["cls1"] == "noxious"), ("gentle (GPT-5.2)", B_["cls1"] == "gentle"), ("deceased (GPT-5.2)", B_["dec1"] == 1)]:
    d = B_[mask]; both_ = d.dropna(subset=["neg1", "neg2"])
    print(f"  atmosphere at {nm:20s}: rated by GPT-5.2 {int(d['neg1'].notna().sum())}, by second coder {int(d['neg2'].notna().sum())}, both {len(both_)}; "
          f"negative/not agreement when both rated {100 * (both_['neg1'] == both_['neg2']).mean():.1f}%")

Set B: 474 locations in 69 dreams


atmosphere valence incl. not coded                      kappa = 0.30 [0.24-0.37]  agreement 48.9%  (n = 474, dreams = 69)


negative vs not, both rated                             kappa = 0.89 [0.79-0.97]  agreement 96.8%  (n = 156, dreams = 59)


deceased person at the location                         kappa = 0.79 [0.66-0.94]  agreement 97.9%  (n = 474, dreams = 69)


animal class at the location (none/gentle/noxious/both) kappa = 0.70 [0.60-0.79]  agreement 92.8%  (n = 474, dreams = 69)
cls2     gentle  none  noxious
cls1                          
both          1     0        0
gentle       27    13        0
none          2   394        3
noxious       0    15       19

P1 within set B (all 27 noxious dreams, 27 of 81 gentle dreams):


  GPT-5.2 animals, GPT-5.2 atmosphere              noxious 84.8% (33 locations, 27 dreams), gentle 63.6% (33 locations, 31 dreams); OR 3.36 [0.82-13.82], one-sided p = 0.0462
  second-coder animals, GPT-5.2 atmosphere         noxious 86.4% (22 locations, 16 dreams), gentle 65.2% (23 locations, 22 dreams); OR 3.35 [0.48-23.36], one-sided p = 0.111
  GPT-5.2 animals, second-coder atmosphere         noxious 90.5% (21 locations, 19 dreams), gentle 66.7% (15 locations, 15 dreams); OR 4.87 [0.74-32.07], one-sided p = 0.0497
  second-coder animals, second-coder atmosphere    noxious 92.9% (14 locations, 10 dreams), gentle 80.0% (10 locations, 10 dreams); OR 3.28 [0.22-50.16], one-sided p = 0.196

P3: deceased locations
  GPT-5.2 deceased locations: 22 in 18 dreams; second coder: 28 in 16 dreams; both: 20
  GPT-5.2 deceased, GPT-5.2 atmosphere             deceased 45.5% (22 locations, 18 dreams) vs living known (GPT-5.2) 68.8%; OR 0.35 [0.14-0.85], one-sided p = 0.0104
  second-coder deceased,

### Findings — R3

**Detection.**
- **Deceased person at the location: κ = 0.79.** GPT-5.2 found 22 locations, the second coder 28, and 20 are shared.
- **Animal class: κ = 0.70.** The pre-registered word lists, applied to the extracted text, flag 34 noxious-animal locations. The second coder found no living noxious animal at 15 of them, and no living gentle animal at 13 of the 40 gentle ones. The sources of these false positives are:
  - similes ("like rats");
  - word fragments ("worm hole");
  - animals that are feared but absent ("worried about sharks");
  - animal-headed beings ("ant-headed dream makers");
  - models and pictures.

**Atmosphere at the test locations.** Where both coders rated the atmosphere, negative-vs-not agreement is 100% at noxious-animal locations (21), 93.3% at gentle ones (15) and 100% at deceased-person locations (13).

**The pre-registered verdicts under the second coder.**

| Test | GPT-5.2 codes | Second-coder detection, GPT-5.2 atmosphere | Second-coder detection and atmosphere |
|---|---|---|---|
| P1, within set B | OR 3.36 (0.82–13.82), one-sided p = 0.046 | OR 3.35 (0.48–23.36), p = 0.11 | OR 3.28 (0.22–50.16), p = 0.20 |
| P3 | OR 0.35 (0.14–0.85), p = 0.010 | OR 0.35 (0.14–0.87), p = 0.012 | OR 0.19 (0.06–0.60), p = 0.002 (GPT-5.2's living-known group) |

**P3.** Robust. The effect and its significance survive a different detection of the deceased and a different coder of atmosphere.

**P1.**
- **Robust:** the size of the effect. The OR is about 3.3 under every combination of coders.
- **Fragile:** its precision.
  - Removing the word-list false positives leaves 22 noxious locations in 16 dreams, and the one-sided p rises to 0.11.
  - Set B holds only 27 of the 81 gentle-animal dreams. The registered test on the full sample (notebook 07) has more gentle locations, and gave OR 4.08, p = 0.012.
- **Verdict:** P1 stands as a **hit as registered**, with a measured caveat. Its significance depends on a detector with a 44% false-positive rate for noxious animals. A replication should detect animals with a coder, not a word list.

**The archived claim.** "Extraction reliability validated through manual review" has no record in the repository. This notebook is the first measurement.